In [ ]:
import os
import subprocess

# Construct the full path to SwarmUI
swarmui_full_path = os.path.join(os.environ['SWARMPATH'], 'SwarmUI')

# Change directory to SwarmUI installation path reliably
try:
    os.chdir(swarmui_full_path)
    print(f"Changed directory to: {os.getcwd()}")
except FileNotFoundError:
    print(f"Error: SwarmUI directory not found at {swarmui_full_path}. Please ensure it is cloned correctly.")
    # Exit or raise error if directory change fails
    raise

# (Colab stupid-proofing: aggressive git ultraforce pull)
# These commands require being in a valid git repository
print("Updating SwarmUI repository...")
!git fetch --all
!git clean -fdx # Remove untracked files and directories forcefully
!git reset --hard origin/main # Assuming 'main' is the default branch for mcmonkeyprojects/SwarmUI
!git pull

# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

# Ensure dotnet is in PATH for the launch script and execute it
print("Launching SwarmUI...")
!export DOTNET_ROOT=/root/.dotnet && export PATH=$DOTNET_ROOT:$PATH && bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared

In [ ]:
import os
import shutil

# Dynamically search for the wildcards directory in possible paths
possible_paths = [
    '/content/StableSwarmUI/src/wwwroot/wildcards',
    '/content/StableSwarmUI/src/Assets/wildcards',
    '/content/drive/MyDrive/SwarmUI/src/wwwroot/wildcards',
    '/content/drive/MyDrive/StableSwarmUI/src/wwwroot/wildcards'
]

swarm_wildcards_path = None
for path in possible_paths:
    if os.path.exists(path):
        swarm_wildcards_path = path
        break

if not swarm_wildcards_path:
    # Deeper search fallback
    for root, dirs, files in os.walk('/content'):
        if 'wildcards' in dirs and 'StableSwarmUI' in root:
            swarm_wildcards_path = os.path.join(root, 'wildcards')
            break

print(f"Target wildcards path: {swarm_wildcards_path}")

if swarm_wildcards_path and os.path.exists(swarm_wildcards_path):
    sfw_path = os.path.join(swarm_wildcards_path, 'sfw')
    if os.path.exists(sfw_path):
        print("Found 'sfw' folder. Merging its contents into parent 'wildcards' directory to eliminate warnings...")
        for item in os.listdir(sfw_path):
            source = os.path.join(sfw_path, item)
            destination = os.path.join(swarm_wildcards_path, item)
            if os.path.isfile(source):
                try:
                    shutil.copy2(source, destination)
                    print(f"Copied: {item} -> root wildcards folder")
                except Exception as e:
                    print(f"Failed to copy {item}: {e}")
    else:
        # Let's create the folder and fetch/write mock wildcards or look elsewhere if they aren't generated yet
        print("Could not find 'sfw' subfolder inside wildcards. It might be created on initial run.")
else:
    print("Wildcards folder not found. SwarmUI may need to be initialized or launched first so it downloads default wildcards.")

In [ ]:
import os

# Dynamically locate the active SwarmUI/StableSwarmUI directory
swarm_paths = [
    '/content/drive/MyDrive/SwarmUI',
    '/content/StableSwarmUI',
    '/content/SwarmUI',
    '/content/drive/MyDrive/StableSwarmUI'
]

base_path = None
for path in swarm_paths:
    if os.path.exists(path):
        base_path = path
        break

if base_path:
    wildcards_path = os.path.join(base_path, 'src/wwwroot/wildcards')
    os.makedirs(wildcards_path, exist_ok=True)
    os.makedirs(os.path.join(wildcards_path, 'sfw'), exist_ok=True)
    print(f"Target wildcards folder identified: {wildcards_path}")

    # Structure wildcards as dictionary { filepath: list_of_lines }
    wildcard_templates = {
        'creature_adjectives.txt': [
            "mythical", "robotic", "bioluminescent", "ancient", "spectral",
            "etherial", "armored", "draconic", "crystalline", "shadowy"
        ],
        'creature_types.txt': [
            "beast", "golem", "dragon", "serpent", "humanoid",
            "griffin", "phoenix", "chimera", "construct", "spirit"
        ],
        'creature_anatomy.txt': [
            "glowing eyes and majestic wings",
            "sharp metallic claws and horns",
            "translucent crystalline scales",
            "a crown of blue fire and smoky tail",
            "intricate runic engravings across its chassis"
        ],
        'creature_actions.txt': [
            "roaring into the sky",
            "emerging from a dense fog",
            "channeling ancient magic energy",
            "resting atop a mountain peak",
            "stalking through a glowing forest"
        ],
        'sfw/element.txt': [
            "fire", "water", "earth", "air", "lightning",
            "ice", "light", "shadow", "cosmic energy", "nature"
        ],
        'sfw/landscape.txt': [
            "volcanic peaks", "crystal caverns", "floating sky islands",
            "cyberpunk ruins", "ancient redwood forest", "bioluminescent ocean"
        ],
        'sfw/atmosphere.txt': [
            "mysterious", "epic", "serene", "dark fantasy",
            "cyberpunk", "dreamlike", "cinematic", "ethereal"
        ],
        'sfw/lighting.txt': [
            "volumetric rays", "dramatic chiaroscuro", "neon glow",
            "golden hour", "bioluminescent ambient light", "moonlit"
        ],
        'sfw/particle_effect.txt': [
            "floating embers", "glowing runes", "swirling stardust",
            "electrical sparks", "falling autumn leaves", "misty vapor"
        ],
        'sfw/genre.txt': [
            "high fantasy", "science fiction", "steampunk",
            "dark fantasy", "cosmic horror", "mythological"
        ],
        'sfw/art_style.txt': [
            "detailed digital illustration", "concept art paint over",
            "octane render 3D", "unreal engine 5 scene", "fine oil painting"
        ],
        'sfw/camera.txt': [
            "extreme close-up", "wide-angle cinematic shot",
            "isometric view", "low-angle dramatic shot", "macro photography"
        ]
    }

    # Write out each wildcard template
    for rel_path, content in wildcard_templates.items():
        full_file_path = os.path.join(wildcards_path, rel_path)
        os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
        with open(full_file_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(content) + '\n')
        print(f"Successfully generated: {rel_path} ({len(content)} entries)")
else:
    print("Could not automatically locate your SwarmUI installation directory. Please double-check your pathing.")

In [ ]:
import os
import shutil

# Detect active SwarmUI base path
# Prioritize the SWARMPATH set in previous cells (which should be in Drive)
swarm_paths = [
    os.path.join(os.environ.get('SWARMPATH', '/content'), 'SwarmUI'),
    '/content/StableSwarmUI',
    '/content/drive/MyDrive/SwarmUI',
    '/content/SwarmUI',
    '/content/drive/MyDrive/StableSwarmUI'
]

base_path = None
for path in swarm_paths:
    # Check for the existence of a key subdirectory to confirm it's a SwarmUI installation
    if os.path.exists(os.path.join(path, 'src', 'wwwroot')):
        base_path = path
        break

if base_path:
    wildcards_path = os.path.join(base_path, 'src/wwwroot/wildcards')

    # Purge existing wildcard directory if it exists to perform a clean database rebuild
    if os.path.exists(wildcards_path):
        print(f"Purging old wildcard database at: {wildcards_path}...")
        shutil.rmtree(wildcards_path)

    os.makedirs(wildcards_path, exist_ok=True)
    os.makedirs(os.path.join(wildcards_path, 'sfw'), exist_ok=True)
    print(f"Created fresh target wildcards directory at: {wildcards_path}")

    # Define the renewed and comprehensive database
    wildcard_database = {
        'creature_adjectives.txt': [
            "mythical", "robotic", "bioluminescent", "ancient", "spectral",
            "ethereal", "armored", "draconic", "crystalline", "shadowy",
            "celestial", "corrupted", "frostbound", "magmatic", "clockwork"
        ],
        'creature_types.txt': [
            "beast", "golem", "dragon", "serpent", "humanoid",
            "griffin", "phoenix", "chimera", "construct", "spirit",
            "colossus", "leviathan", "wraith", "wyvern", "sentinel"
        ],
        'creature_anatomy.txt': [
            "glowing eyes and majestic wings",
            "sharp metallic claws and horns",
            "translucent crystalline scales",
            "a crown of blue fire and smoky tail",
            "intricate runic engravings across its chassis",
            "overgrown mossy armor plating",
            "crackling electrical nodes running down its spine"
        ],
        'creature_actions.txt': [
            "roaring into the sky",
            "emerging from a dense fog",
            "channeling ancient magic energy",
            "resting atop a mountain peak",
            "stalking through a glowing forest",
            "ascending into a storm cloud",
            "guarding an ancient golden chest"
        ],
        'sfw/element.txt': [
            "fire", "water", "earth", "air", "lightning",
            "ice", "light", "shadow", "cosmic energy", "nature",
            "magma", "plasma", "gravity", "time", "spectral essence"
        ],
        'sfw/landscape.txt': [
            "volcanic peaks", "crystal caverns", "floating sky islands",
            "cyberpunk ruins", "ancient redwood forest", "bioluminescent ocean",
            "steaming geothermal geysers", "windswept desert dunes", "overgrown stone temples"
        ],
        'sfw/atmosphere.txt': [
            "mysterious", "epic", "serene", "dark fantasy",
            "cyberpunk", "dreamlike", "cinematic", "ethereal",
            "majestic", "ominous", "nostalgic", "surreal"
        ],
        'sfw/lighting.txt': [
            "volumetric rays", "dramatic chiaroscuro", "neon glow",
            "golden hour", "bioluminescent ambient light", "moonlit",
            "sunshafts filtering through trees", "intense rim lighting"
        ],
        'sfw/particle_effect.txt': [
            "floating embers", "glowing runes", "swirling stardust",
            "electrical sparks", "falling autumn leaves", "misty vapor",
            "shattered glass shards", "floating cherry blossom petals"
        ],
        'sfw/genre.txt': [
            "high fantasy", "science fiction", "steampunk",
            "dark fantasy", "cosmic horror", "mythological",
            "dieselpunk", "cyber-noir", "solarpunk"
        ],
        'sfw/art_style.txt': [
            "detailed digital illustration", "concept art paint over",
            "octane render 3D", "unreal engine 5 scene", "fine oil painting",
            "watercolor and ink sketching", "retro anime style cel shading"
        ],
        'sfw/camera.txt': [
            "extreme close-up", "wide-angle cinematic shot",
            "isometric view", "low-angle dramatic shot", "macro photography",
            "gopro action capture", "satellite orbital view"
        ],
        'templates.txt': [
            "A <creature_adjectives> <creature_types> with <creature_anatomy>, <creature_actions> in <sfw/landscape>, <sfw/atmosphere> atmosphere, <sfw/lighting> lighting, <sfw/particle_effect> particle effects, <sfw/genre> style, <sfw/art_style>, <sfw/camera>",
            "Epic <sfw/art_style> of a <creature_adjectives> <creature_types>, elemental power of <sfw/element>, dramatic <sfw/lighting>, cinematic <sfw/camera>",
            "A majestic <creature_types> showing <creature_anatomy>, <creature_actions>, <sfw/particle_effect> details, set in a surreal <sfw/landscape>, <sfw/genre> genre",
            "Close-up <sfw/camera> of <creature_adjectives> <creature_types> infused with <sfw/element>, glowing runes, <sfw/art_style>"
        ]
    }

    # Generate new database files and synchronize them
    for rel_path, items in wildcard_database.items():
        full_file_path = os.path.join(wildcards_path, rel_path)
        os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
        with open(full_file_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(items) + '\n')

        # If it's a file inside a subdirectory, copy a duplicate to the root directory
        # to support flat format queries/compatibility issues with some nodes
        if '/' in rel_path or '\\' in rel_path:
            flat_filename = os.path.basename(rel_path)
            flat_target = os.path.join(wildcards_path, flat_filename)
            shutil.copy2(full_file_path, flat_target)

    print("\nAll wildcard database files (including the templates wildcard) have been completely rebuilt and mirrored to the root directory.")
else:
    print("Could not automatically locate your SwarmUI directory. Please check that SwarmUI is installed correctly. Make sure SwarmUI is cloned into '/content/drive/MyDrive/SwarmUI'.")

In [ ]:
# Pre-install ComfyUI dependencies globally to resolve the 'ModuleNotFoundError: No module named filelock' loop
!pip install filelock Pillow torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# Also install the requirements directly from the ComfyUI directory if it exists
import os
comfy_reqs = '/content/StableSwarmUI/dlbackend/ComfyUI/requirements.txt'
if os.path.exists(comfy_reqs):
    print("Installing requirements from ComfyUI requirements.txt...")
    !pip install -r {comfy_reqs}
else:
    print("ComfyUI requirements.txt not found yet. It will be created when SwarmUI fully runs or downloads it.")

### SwarmUI Installation Path Selection
Please select your preferred installation path for SwarmUI. Choosing 'Google Drive (Persistent)' will mount your Google Drive and install SwarmUI there, allowing your data to persist across sessions. 'Temporary (/content/)' will install SwarmUI in the Colab instance's temporary storage, which will be lost when the session ends.

In [ ]:
# This cell was identified as cloning a conflicting SwarmUI version (Stability-AI/StableSwarmUI) to a temporary path, which caused `ModuleNotFoundError` for `filelock` and interfered with the intended `mcmonkeyprojects/SwarmUI` installation. Its content has been cleared to ensure consistent installation of the intended SwarmUI to Google Drive.

In [ ]:
import os

drive_checkpoints = '/content/drive/MyDrive/SwarmColab/Models/checkpoints'
drive_loras = '/content/drive/MyDrive/SwarmColab/Models/loras'

swarm_checkpoints = '/content/StableSwarmUI/Models/Stable-Diffusion'
swarm_loras = '/content/StableSwarmUI/Models/Lora'

print("=== Persistent Google Drive Model Files ===")
if os.path.exists(drive_checkpoints):
    checkpoints = os.listdir(drive_checkpoints)
    print(f"Checkpoints in Drive ({len(checkpoints)}): {checkpoints}")
else:
    print("Drive checkpoints folder does not exist.")

if os.path.exists(drive_loras):
    loras = os.listdir(drive_loras)
    print(f"Loras in Drive ({len(loras)}): {loras}")
else:
    print("Drive loras folder does not exist.")

print("\n=== Active SwarmUI Directory Symlinks ===")
for name, path in [("Checkpoints", swarm_checkpoints), ("Loras", swarm_loras)]:
    if os.path.islink(path):
        target = os.readlink(path)
        print(f"{name} symlink exists and points to: {target}")
        try:
            resolved_files = os.listdir(path)
            print(f"  Resolved files through symlink ({len(resolved_files)}): {resolved_files}")
        except Exception as e:
            print(f"  Error reading files through symlink: {e}")
    elif os.path.exists(path):
        print(f"{name} path exists but is a standard directory (not a symlink).")
    else:
        print(f"{name} path does not exist.")

# New Section

In [ ]:
#### PICK ONE: *OR* this node for install in google drive
from google.colab import drive
drive.mount('/content/drive')
SWARMPATH = '/content/drive/MyDrive'

# Colab breaks venvs, and doesn't save anything valid to drive, so just screw it do a global install of pip reqs if we used a comfy backend in the drive
!if [[ -f "$SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt" ]]; then rm -rf $SWARMPATH/SwarmUI/dlbackend/ComfyUI/venv/; pip install -r $SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt; fi

In [ ]:
import os
import shutil

# Install dotnet dependencies
!wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Set DOTNET_ROOT and add to PATH to ensure dotnet is found for current session and sub-shells
os.environ['DOTNET_ROOT'] = '/root/.dotnet'
os.environ['PATH'] = os.environ['DOTNET_ROOT'] + ':' + os.environ['PATH']
!export DOTNET_ROOT=/root/.dotnet
!export PATH=$DOTNET_ROOT:$PATH

# Install Cloudflared
!wget -nc https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Use the chosen path from previous cells (SWARMPATH is expected to be '/content/drive/MyDrive')
os.environ['SWARMPATH'] = SWARMPATH
# Ensure we are in the base SWARMPATH before attempting SwarmUI operations
%cd $SWARMPATH

# Colab breaks venv, so, tell swarm to not make a venv
os.environ['SWARM_NO_VENV'] = 'true'

# Download SwarmUI (mcmonkeyprojects/SwarmUI)
url = "https://github.com/mcmonkeyprojects/SwarmUI"
swarmui_target_path = os.path.join(SWARMPATH, 'SwarmUI')

# Check if SwarmUI directory exists and is a valid git repository
if os.path.exists(swarmui_target_path):
    # If it exists, but isn't a valid git repo, remove and re-clone
    if not os.path.isdir(os.path.join(swarmui_target_path, '.git')):
        print(f"Existing SwarmUI at {swarmui_target_path} is not a valid git repository. Removing and re-cloning...")
        shutil.rmtree(swarmui_target_path)
        !git clone $url $swarmui_target_path
    else:
        print(f"SwarmUI already exists at {swarmui_target_path} and is a valid git repository, skipping clone.")
else:
    print(f"Cloning SwarmUI into {swarmui_target_path}...")
    !git clone $url $swarmui_target_path

# Verify the SwarmUI directory exists after cloning/checking
if not os.path.exists(swarmui_target_path):
    raise FileNotFoundError(f"SwarmUI directory not found at {swarmui_target_path} after cloning attempt.")

In [ ]:
# Alright, launch it! Watch the output for a Cloudflare URL
%cd $SWARMPATH/SwarmUI/

# (Colab stupid-proofing: aggressive git ultraforce pull)
!git fetch --all
!git reset --hard origin/main # Assuming 'main' is the default branch for mcmonkeyprojects/SwarmUI
!git pull

# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared

In [ ]:
print('Checking for running SwarmUI process...')
!ps aux | grep "SwarmUI"

In [ ]:
# Establish the environment and execute the linux launcher for SwarmUI
import os
import subprocess
import re

# Configure Swarm to use global pip packages rather than a local venv inside Colab
os.environ['SWARM_NO_VENV'] = 'true'

# Point to active installation path
swarm_dir = '/content/StableSwarmUI' if os.path.exists('/content/StableSwarmUI') else '/content/drive/MyDrive/SwarmUI'
os.chdir(swarm_dir)
print(f"Navigated to SwarmUI directory: {swarm_dir}")

cmd = ["bash", "./launch-linux.sh", "--launch_mode", "none", "--cloudflared-path", "cloudflared"]

print("[SwarmUI] Starting the background server process...")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Display output logs in real-time and look for the TryCloudflare link
url_captured = False
for line in process.stdout:
    print(line, end="")
    if not url_captured and "trycloudflare.com" in line:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            print("\n" + "="*60)
            print(f">>> [SWARMUI ACTIVE TUNNEL URL]: {tunnel_url} <<<")
            print("="*60 + "\n")
            url_captured = True